In [29]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import sys
import os
import json
import asyncio
from pathlib import Path
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader



# Add the src directory to the Python path
sys.path.append(str(Path().resolve().parent / "src"))
import nest_asyncio

nest_asyncio.apply()

with open((Path().resolve().parent / "hf_token.json").as_posix()) as f:
    HF_TOKEN: str = json.load(f)["HUGGING_FACE_TOKEN"]

LLM Call

In [22]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm: HuggingFaceInferenceAPI = HuggingFaceInferenceAPI(
    model_name="deepseek-ai/DeepSeek-V3-0324",
    token=HF_TOKEN,
    provider="auto",  # this will use the best provider available
    temperature=0.1,
)

print(llm.complete(prompt="What is the capital of France?").text)

The capital of France is **Paris**. It is known for its iconic landmarks such as the Eiffel Tower, the Louvre Museum, and the Arc de Triomphe.  

Would you like more details about Paris or France? 😊


Embedding Model

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceInferenceAPIEmbedding

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


embed_model = HuggingFaceInferenceAPIEmbedding(model_name=MODEL_NAME, token=HF_TOKEN)
# --- 2. Get Embeddings ---
sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "Exploring the solar system is a great adventure.",
]

# Use the proper public method for getting embeddings
embeddings = []
for sentence in sentences:
    embedding = embed_model.get_text_embedding(sentence)
    embeddings.append(embedding)

print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding dimension: {len(embeddings[0]) if embeddings else 'N/A'}")

C:\Users\Valentin\AppData\Local\Temp\ipykernel_1916\2559246248.py:5: DeprecationWarning: Call to deprecated class HuggingFaceInferenceAPIEmbedding. (Deprecated in favor of `HuggingFaceInferenceAPIEmbedding` from `llama-index-embeddings-huggingface-api` which should be used instead.)
  embed_model = HuggingFaceInferenceAPIEmbedding(model_name=MODEL_NAME, token=HF_TOKEN)


Generated 3 embeddings
Embedding dimension: 384


RAG Part

In [31]:
from retrieval.rag_helper import RAGHelper

In [ ]:
rag_helper = RAGHelper(llm=llm, embed_model=embed_model, hf_token=HF_TOKEN)

document_path: str = (Path().resolve().parent / "docs").as_posix()

doc_lst = rag_helper.load_documents(directory_path=document_path)

rag_helper.build_index(documents=doc_lst)

query_engine = rag_helper.get_query_engine()

In [ ]:
query_engine.query(
    "De quelle ville est issu ce cahier des charges techniques et environementales ?"
)

Response(response='Ce cahier des charges techniques et environnementales est issu de la ville de Lyon.', source_nodes=[NodeWithScore(node=TextNode(id_='c2ce5b19-1aeb-4abf-90f8-3d8f337e9a85', embedding=None, metadata={'page_label': '2', 'file_name': 'cahier-charges-techs-environnementales.pdf', 'file_path': 'C:\\Users\\Valentin\\Projects\\DS_Project_Test\\docs\\cahier-charges-techs-environnementales.pdf', 'file_type': 'application/pdf', 'file_size': 7356447, 'creation_date': '2025-10-11', 'last_modified_date': '2025-10-11'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='e060a42b-81d1-4cb0-b841-cba3ed1d807b', node_type='4', metadata={'page_label': '2', 'file_name': 'cahier-charges-techs-environnementales.pd